#### Hypothesis 2: There are specific categories that are causing decline in median order value

In [1]:
import pandas as pd 
import numpy as np 

In [2]:
# loading the data
data = pd.read_parquet("../../data/processed/superstore/data.parquet")
orders = pd.read_parquet("../../data/processed/superstore/orders.parquet")
orders_agg = pd.read_parquet("../../data/processed/superstore/orders_agg.parquet")
dates = pd.read_parquet("../../data/processed/superstore/dates.parquet")
product = pd.read_parquet("../../data/processed/superstore/product.parquet")

##### Revenue trend by category

In [4]:
# revenue by year and category | Year-over-year growth
category_revenue =  orders[["Product ID", "Order Date","Sales"]].merge(
    dates[["date","year","month_num","month"]], 
    left_on="Order Date", right_on="date").merge(
        product[["product_id","Product ID", "Category", 
                 "Sub-Category"]], 
        on="Product ID" )[["year","Category", "Sales"]].groupby(
            ["year","Category"]
        )["Sales"].sum().reset_index().pivot(
            index="Category",
            columns = "year",
            values = "Sales"
        )

category_revenue.pct_change(axis=1).round(2)

year,2014,2015,2016,2017
Category,,,,
Furniture,NaN,0.08,0.17,0.06
Office Supplies,NaN,-0.09,0.35,0.34
Technology,NaN,-0.05,0.38,0.20


- Technology experienced slower revenue growth in 2017
- Furniture showed relatively weak revenue growth
- Office supplies even though dropped a bit once in 2015 but further on maintained a strong revenue growth

This suggests that not every category contributed equally to revenue performance

##### Quantity growth by Category

In [7]:
# quantity trend by Category

category_growth = orders[["Product ID", "Order Date","Quantity"]].merge(
    dates[["date","year","month_num","month"]], 
    left_on="Order Date", right_on="date").merge(
        product[["product_id","Product ID", "Category", 
                 "Sub-Category"]], 
        on="Product ID" )[["year","Category", "Quantity"]].groupby(
            ["year","Category"]
        )["Quantity"].sum().reset_index().pivot(
            index="Category",
            columns = "year",
            values = "Quantity"
        )

category_growth.pct_change(axis=1).round(2)

year,2014,2015,2016,2017
Category,,,,
Furniture,NaN,0.10,0.23,0.11
Office Supplies,NaN,0.03,0.28,0.28
Technology,NaN,0.08,0.15,0.38


In [17]:
category_revenue.pct_change(axis=1).round(2).merge(
    category_growth.pct_change(axis=1).round(2), 
    suffixes=("_Revenue","_Quantity"),
    left_on="Category",
    right_on="Category")[[
                          "2015_Revenue","2015_Quantity",
                          "2016_Revenue","2016_Quantity",
                          "2017_Revenue","2017_Quantity"]]

year,2015_Revenue,2015_Quantity,2016_Revenue,2016_Quantity,2017_Revenue,2017_Quantity
Category,,,,,,
Furniture,0.08,0.10,0.17,0.23,0.06,0.11
Office Supplies,-0.09,0.03,0.35,0.28,0.34,0.28
Technology,-0.05,0.08,0.38,0.15,0.20,0.38


- The ratio of change in quantity and revenue is not proportional.

##### Furniture: 
- Across three years the quantity growth exceeded revenue growth
- Business sold more furniture products each year, but revenue increased more slowly than sales volume.
    - This suggests that additional sales generated less revenue  per unit or per order than before. Possible reasons include discounting, lower-priced product purchases or a shift toward less expensive furniture items.

##### Office Supplies: 
- Expereinced an initial decline but recovered substantially. Revenue growth outpaced quantity growth after 2015, suggesting improved revenue generation per unit sold
- This category generated increasing revenue from each additional unit sold.
- Therefore, it is unlikely to be a major contributor to the overall decline in Median Order Value

##### Technology: 
- Showed inconsistent growth, initial decline in revenue and then increase. However, quantity growth is not at all proportional to revenue growth. Higher sales volume did proportionally translate into higher revenue.
- Customers bought more tech products, but those purchases generated proportionally less revneue.
    - This pattern reflects increased sales of lower-priced tech products, heavier disocunting or changes in the product mix.


### RESULSTS
It partially supports the hypothesis
- Technology provides the strongest evidence supporting the hypothesis. Sales volume increased much faster than revenue, suggesting declining revnenue generated per sale or oorder.
- Furniture also shows a similar pattern, although the gap between quantity and revenue growth is smaller.
- Office supplies does not support the hypothesis because revenue growth receovered and generally outpaced quantity growth after 2015.

In [21]:
# Median order value by Category
category_mov=  orders[["Product ID", "Order Date","Sales"]].merge(
    dates[["date","year","month_num","month"]], 
    left_on="Order Date", right_on="date").merge(
        product[["product_id","Product ID", "Category", 
                 "Sub-Category"]], 
        on="Product ID" )[["year","Category", "Sales"]].groupby(
            ["year","Category"]
        )["Sales"].median().reset_index().pivot(
            index="Category",
            columns = "year",
            values = "Sales"
        )
category_mov.pct_change(axis=1).round(2)

year,2014,2015,2016,2017
Category,,,,
Furniture,NaN,0.12,-0.06,-0.12
Office Supplies,NaN,0.11,0.03,-0.08
Technology,NaN,-0.00,-0.12,-0.10


This confirms that technology and some extent to furniture as well show declining median order value by category.
Hence, these categories are contributing overall decline in revenue performance